### 07b. Time-Domain Feature Extraction & Feature Quality Assessment

#### Objective

This notebook extracts time-domain features from the preprocessed PADS smartwatch movement recordings generated during the signal preprocessing stage.

For each participant, motor task, and wrist recording, descriptive features are computed from the accelerometer and gyroscope X, Y, and Z axes together with their corresponding vector magnitudes. The extracted features include mean, median, standard deviation, minimum, maximum, range, interquartile range (IQR), root mean square (RMS), and signal energy.

The resulting feature table is then evaluated to verify its quality before subsequent feature engineering and machine-learning analyses. The assessment includes:

- Feature completeness
- Missing and infinite values
- Constant features
- Low-variance features and invalid numerical values

The generated outputs include:

- Time-domain feature task aware table
- Task aware feature quality report
- Task aware feature quality summary
- List of constant and low-variance features.

In [1]:
# ===========================================================
# Libraries and project paths
# ===========================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# ===============================================================
# Locate project root
# ===============================================================

cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_ROOT = cwd

elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent

else:
    raise FileNotFoundError(
        "Could not locate the project root containing data/ and src/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ====================================================
# Import project functions
# ====================================================

from src.features.time_features import (
    extract_time_features,
)

# ====================================================
# Project paths
# ====================================================

NPZ_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "preprocessed_signals"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "time_domain_features"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Recording-level feature table
RECORDING_TIME_FEATURES_OUTPUT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "time_domain_features_recording_level.csv"
)

# Recording-level quality outputs
RECORDING_FEATURE_QUALITY_REPORT_OUTPUT = (
    OUTPUT_DIR / "recording_feature_quality_report.csv"
)

FEATURE_QUALITY_SUMMARY_OUTPUT = (
    OUTPUT_DIR / "feature_quality_summary.csv"
)

RECORDING_CONSTANT_FEATURES_OUTPUT = (
    OUTPUT_DIR / "recording_constant_features.csv"
)

RECORDING_INVALID_FEATURES_OUTPUT = (
    OUTPUT_DIR / "recording_invalid_features.csv"
)

RECORDING_LOW_VARIANCE_FEATURES_OUTPUT = (
    OUTPUT_DIR / "recording_low_variance_features.csv"
)

FEATURE_COMPLETENESS_OUTPUT = (
    OUTPUT_DIR / "feature_completeness.csv"
)

FEATURE_COMPLETENESS_SUMMARY_OUTPUT = (
    OUTPUT_DIR / "feature_completeness_summary.csv"
)

# Configurable threshold
# Features with variance > 0 and below this value will be flagged.
LOW_VARIANCE_THRESHOLD = 1e-6

# Task-aware participant-level feature table
TIME_FEATURES_OUTPUT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "time_domain_features.csv"
)

# Task-aware quality outputs
TASK_AWARE_FEATURE_QUALITY_REPORT_OUTPUT = (
    OUTPUT_DIR
    / "task_aware_feature_quality_report.csv"
)

# Task-aware quality summary
TASK_AWARE_FEATURE_QUALITY_SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "task_aware_feature_quality_summary.csv"
)

TASK_AWARE_CONSTANT_FEATURES_OUTPUT = (
    OUTPUT_DIR
    / "task_aware_constant_features.csv"
)

TASK_AWARE_INVALID_FEATURES_OUTPUT = (
    OUTPUT_DIR
    / "task_aware_invalid_features.csv"
)

TASK_AWARE_LOW_VARIANCE_FEATURES_OUTPUT = (
    OUTPUT_DIR
    / "task_aware_low_variance_features.csv"
)

# ===================================================
# Expected dataset structure
# ===================================================

EXPECTED_PARTICIPANTS = 469
EXPECTED_RECORDINGS_PER_PARTICIPANT = 22
EXPECTED_TOTAL_RECORDINGS = (
    EXPECTED_PARTICIPANTS
    * EXPECTED_RECORDINGS_PER_PARTICIPANT
)

EXPECTED_TIME_FEATURES = 72

print("Project root:", PROJECT_ROOT)
print("Processed signal directory:", NPZ_DIR)
print("Output directory:", OUTPUT_DIR)
print("Low-variance threshold:", LOW_VARIANCE_THRESHOLD)

Project root: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease
Processed signal directory: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\preprocessed_signals
Output directory: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\tables\time_domain_features
Low-variance threshold: 1e-06


#### 1. Confirm Input Availability

The participant-level NPZ files generated by the signal preprocessing pipeline are verified before feature extraction.

In [2]:
# ===================================
# Confirm input availability
# ===================================

if not NPZ_DIR.exists():
    raise FileNotFoundError(
        f"Could not locate {NPZ_DIR}."
    )

npz_files = sorted(
    NPZ_DIR.glob("*_preprocessed.npz")
)

if not npz_files:
    raise FileNotFoundError(
        f"No participant NPZ files were found in {NPZ_DIR}."
    )

print("Participant NPZ files found:", len(npz_files))
print("First file:", npz_files[0].name)
print("Last file:", npz_files[-1].name)

assert len(npz_files) == EXPECTED_PARTICIPANTS, (
    f"Expected {EXPECTED_PARTICIPANTS} participant files, "
    f"but found {len(npz_files)}."
)

Participant NPZ files found: 469
First file: 001_preprocessed.npz
Last file: 469_preprocessed.npz


#### 2. Inspect Dataset Structure

In [3]:
# ====================================
# Inspect dataset structure
# ====================================

sample_npz = npz_files[0]

with np.load(
    sample_npz,
    allow_pickle=False,
) as data:

    npz_keys = data.files

print("Sample participant file:", sample_npz.name)
print("Number of entries stored:", len(npz_keys))

print("\nNPZ keys:\n")

for key in npz_keys:
    print(key)

Sample participant file: 001_preprocessed.npz
Number of entries stored: 24

NPZ keys:

__columns__
__recording_keys__
CrossArms__LeftWrist
CrossArms__RightWrist
DrinkGlas__LeftWrist
DrinkGlas__RightWrist
Entrainment__LeftWrist
Entrainment__RightWrist
HoldWeight__LeftWrist
HoldWeight__RightWrist
LiftHold__LeftWrist
LiftHold__RightWrist
PointFinger__LeftWrist
PointFinger__RightWrist
Relaxed__LeftWrist
Relaxed__RightWrist
RelaxedTask__LeftWrist
RelaxedTask__RightWrist
StretchHold__LeftWrist
StretchHold__RightWrist
TouchIndex__LeftWrist
TouchIndex__RightWrist
TouchNose__LeftWrist
TouchNose__RightWrist


In [4]:
# ====================================
# Inspect one metadata
# ====================================

with np.load(
    sample_npz,
    allow_pickle=False,
) as data:

    processed_columns = (
        data["__columns__"]
        .astype(str)
        .tolist()
    )

    recording_keys = (
        data["__recording_keys__"]
        .astype(str)
        .tolist()
    )

print("Processed columns:")
print(processed_columns)

print("\nNumber of recordings:", len(recording_keys))
print("First recording key:", recording_keys[0])
print("Last recording key:", recording_keys[-1])

assert len(recording_keys) == EXPECTED_RECORDINGS_PER_PARTICIPANT, (
    f"Expected {EXPECTED_RECORDINGS_PER_PARTICIPANT} recordings, "
    f"but found {len(recording_keys)}."
)

Processed columns:
['Time', 'Accelerometer_X', 'Accelerometer_Y', 'Accelerometer_Z', 'Gyroscope_X', 'Gyroscope_Y', 'Gyroscope_Z', 'Acc_Magnitude', 'Gyro_Magnitude']

Number of recordings: 22
First recording key: CrossArms__LeftWrist
Last recording key: TouchNose__RightWrist


In [5]:
# ====================================
# Inspecting a recording
# ====================================

# Select the first recording stored in the participant file
sample_recording_key = recording_keys[0]

with np.load(
    sample_npz,
    allow_pickle=False,
) as data:

    sample_array = data[sample_recording_key]
    sample_columns = (
        data["__columns__"]
        .astype(str)
        .tolist()
    )

sample_signal = pd.DataFrame(
    sample_array,
    columns=sample_columns,
)

print("patient_id:", sample_npz.stem.replace("_preprocessed", ""))
print("recording key:", sample_recording_key)
print("Signal shape:", sample_signal.shape)
print("Columns:", sample_signal.columns.tolist())

sample_signal.head()

patient_id: 001
recording key: CrossArms__LeftWrist
Signal shape: (976, 9)
Columns: ['Time', 'Accelerometer_X', 'Accelerometer_Y', 'Accelerometer_Z', 'Gyroscope_X', 'Gyroscope_Y', 'Gyroscope_Z', 'Acc_Magnitude', 'Gyro_Magnitude']


,Time,Accelerometer_X,Accelerometer_Y,Accelerometer_Z,Gyroscope_X,Gyroscope_Y,Gyroscope_Z,Acc_Magnitude,Gyro_Magnitude
0,0.000000,0.128851,0.208589,-0.051914,0.618479,0.166856,-0.285923,0.250613,0.701505
1,0.009971,0.128674,0.197031,-0.057510,0.676682,0.249149,-0.339617,0.242251,0.797066
2,0.019953,0.133072,0.191920,-0.054687,0.717691,0.341025,-0.381673,0.239859,0.881506
3,0.029947,0.145011,0.196137,-0.050303,0.773790,0.420087,-0.436407,0.249055,0.982687
4,0.039948,0.155719,0.202804,-0.044261,0.854607,0.477981,-0.488896,0.259494,1.094458


#### 3. Extract Time-Domain Features

In [6]:
# ===============================================
# Test feature extraction on one recording
# ===============================================

sample_features = extract_time_features(
    sample_signal
)

print("Number of extracted features:", len(sample_features))

assert len(sample_features) == EXPECTED_TIME_FEATURES, (
    f"Expected {EXPECTED_TIME_FEATURES} time-domain features, "
    f"but found {len(sample_features)}."
)

sample_features

Number of extracted features: 72


Accelerometer_X_Mean        -0.000278
Accelerometer_X_Median      -0.000864
Accelerometer_X_Std          0.086120
Accelerometer_X_Min         -0.405191
Accelerometer_X_Max          0.395058
Accelerometer_X_Range        0.800248
Accelerometer_X_IQR          0.010461
Accelerometer_X_RMS          0.086077
Accelerometer_X_Energy       7.231351
Accelerometer_Y_Mean        -0.001313
Accelerometer_Y_Median       0.000048
Accelerometer_Y_Std          0.135839
Accelerometer_Y_Min         -0.523505
Accelerometer_Y_Max          0.499106
Accelerometer_Y_Range        1.022610
Accelerometer_Y_IQR          0.017189
Accelerometer_Y_RMS          0.135776
Accelerometer_Y_Energy      17.992600
Accelerometer_Z_Mean        -0.001328
Accelerometer_Z_Median       0.000435
Accelerometer_Z_Std          0.096446
Accelerometer_Z_Min         -0.469866
Accelerometer_Z_Max          0.325920
Accelerometer_Z_Range        0.795786
Accelerometer_Z_IQR          0.013673
Accelerometer_Z_RMS          0.096405
Acceleromete

#### 4. Build Time-Domain Feature Table

The feature-extraction function is applied to every task-wrist recording stored in the 469 participant NPZ files. Each output row contains the participant identifier, motor task, wrist location, recording key, source NPZ file, and the 72 extracted time-domain features.

In [7]:
# ========================================================
# Extract features from all participant recordings
# ========================================================

feature_rows = []
extraction_errors = []

for file_number, npz_file in enumerate(
    npz_files,
    start=1,
):

    participant_id = npz_file.stem.replace(
        "_preprocessed",
        ""
    )

    try:
        with np.load(
            npz_file,
            allow_pickle=False,
        ) as data:

            columns = (
                data["__columns__"]
                .astype(str)
                .tolist()
            )

            participant_recording_keys = (
                data["__recording_keys__"]
                .astype(str)
                .tolist()
            )

            if (
                len(participant_recording_keys)
                != EXPECTED_RECORDINGS_PER_PARTICIPANT
            ):
                raise ValueError(
                    f"Participant {participant_id} contains "
                    f"{len(participant_recording_keys)} recordings "
                    f"instead of {EXPECTED_RECORDINGS_PER_PARTICIPANT}."
                )

            for recording_key in participant_recording_keys:

                if "__" not in recording_key:
                    raise ValueError(
                        f"Unexpected recording key format: {recording_key}"
                    )

                task, wrist = recording_key.split(
                    "__",
                    maxsplit=1,
                )

                signal_array = data[recording_key]

                signal = pd.DataFrame(
                    signal_array,
                    columns=columns,
                )

                # Extract the 72 time-domain features
                feature_series = extract_time_features(
                    signal
                )

                # Store identifiers before feature values
                feature_row = {
                    "patient_id": participant_id,
                    "task": task,
                    "wrist": wrist,
                    "recording_key": recording_key,
                    "source_file": npz_file.name,
                }

                feature_row.update(
                    feature_series.to_dict()
                )

                feature_rows.append(
                    feature_row
                )

    except Exception as exc:

        extraction_errors.append(
            {
                "patient_id": participant_id,
                "source_file": npz_file.name,
                "Error": str(exc),
            }
        )

        print(
            f"Feature extraction failed for "
            f"{npz_file.name}: {exc}"
        )

    if (
        file_number % 50 == 0
        or file_number == len(npz_files)
    ):
        print(
            f"Processed participant files: "
            f"{file_number}/{len(npz_files)}"
        )


Processed participant files: 50/469
Processed participant files: 100/469
Processed participant files: 150/469
Processed participant files: 200/469
Processed participant files: 250/469
Processed participant files: 300/469
Processed participant files: 350/469
Processed participant files: 400/469
Processed participant files: 450/469
Processed participant files: 469/469


#### 5. Create and validate the feature table

In [8]:
# ============================================
# Create the time-domain feature table (recording level)
# ============================================

time_feature_table = pd.DataFrame(
    feature_rows
)

metadata_columns = [
    "patient_id",
    "task",
    "wrist",
    "recording_key",
    "source_file",
]

feature_columns = [
    column
    for column in time_feature_table.columns
    if column not in metadata_columns
]

print("Feature table shape:", time_feature_table.shape)
print("Recordings:", len(time_feature_table))
print("Metadata columns:", len(metadata_columns))
print("Time-domain feature columns:", len(feature_columns))
print("Extraction errors:", len(extraction_errors))

assert len(time_feature_table) == EXPECTED_TOTAL_RECORDINGS, (
    f"Expected {EXPECTED_TOTAL_RECORDINGS} recordings, "
    f"but found {len(time_feature_table)}."
)

assert len(feature_columns) == EXPECTED_TIME_FEATURES, (
    f"Expected {EXPECTED_TIME_FEATURES} feature columns, "
    f"but found {len(feature_columns)}."
)

assert len(extraction_errors) == 0, (
    f"Feature extraction produced {len(extraction_errors)} errors."
)

time_feature_table.head()

Feature table shape: (10318, 77)
Recordings: 10318
Metadata columns: 5
Time-domain feature columns: 72
Extraction errors: 0


,patient_id,task,wrist,recording_key,source_file,Accelerometer_X_Mean,Accelerometer_X_Median,Accelerometer_X_Std,Accelerometer_X_Min,Accelerometer_X_Max,Accelerometer_X_Range,Accelerometer_X_IQR,Accelerometer_X_RMS,Accelerometer_X_Energy,Accelerometer_Y_Mean,Accelerometer_Y_Median,Accelerometer_Y_Std,Accelerometer_Y_Min,Accelerometer_Y_Max,Accelerometer_Y_Range,Accelerometer_Y_IQR,Accelerometer_Y_RMS,Accelerometer_Y_Energy,Accelerometer_Z_Mean,Accelerometer_Z_Median,Accelerometer_Z_Std,Accelerometer_Z_Min,Accelerometer_Z_Max,Accelerometer_Z_Range,Accelerometer_Z_IQR,Accelerometer_Z_RMS,Accelerometer_Z_Energy,Gyroscope_X_Mean,Gyroscope_X_Median,Gyroscope_X_Std,Gyroscope_X_Min,Gyroscope_X_Max,Gyroscope_X_Range,Gyroscope_X_IQR,Gyroscope_X_RMS,Gyroscope_X_Energy,Gyroscope_Y_Mean,Gyroscope_Y_Median,Gyroscope_Y_Std,Gyroscope_Y_Min,Gyroscope_Y_Max,Gyroscope_Y_Range,Gyroscope_Y_IQR,Gyroscope_Y_RMS,Gyroscope_Y_Energy,Gyroscope_Z_Mean,Gyroscope_Z_Median,Gyroscope_Z_Std,Gyroscope_Z_Min,Gyroscope_Z_Max,Gyroscope_Z_Range,Gyroscope_Z_IQR,Gyroscope_Z_RMS,Gyroscope_Z_Energy,Acc_Magnitude_Mean,Acc_Magnitude_Median,Acc_Magnitude_Std,Acc_Magnitude_Min,Acc_Magnitude_Max,Acc_Magnitude_Range,Acc_Magnitude_IQR,Acc_Magnitude_RMS,Acc_Magnitude_Energy,Gyro_Magnitude_Mean,Gyro_Magnitude_Median,Gyro_Magnitude_Std,Gyro_Magnitude_Min,Gyro_Magnitude_Max,Gyro_Magnitude_Range,Gyro_Magnitude_IQR,Gyro_Magnitude_RMS,Gyro_Magnitude_Energy
0,001,CrossArms,LeftWrist,CrossArms__LeftWrist,001_preprocessed.npz,-0.000278,-0.000864,0.086120,-0.405191,0.395058,0.800248,0.010461,0.086077,7.231351,-0.001313,0.000048,0.135839,-0.523505,0.499106,1.022610,0.017189,0.135776,17.992600,-0.001328,0.000435,0.096446,-0.469866,0.325920,0.795786,0.013673,0.096405,9.070935,-0.043856,-0.008570,0.593056,-2.152440,3.150466,5.302906,0.059571,0.594373,344.800173,-0.079353,-0.008092,0.838249,-2.372351,3.295569,5.667920,0.096337,0.841569,691.241211,-0.282019,-0.001827,1.950556,-6.042105,7.007679,13.049784,0.037299,1.969849,3787.177850,0.108274,0.012572,0.153098,0.001195,0.690349,0.689154,0.200304,0.187452,34.294886,1.101626,0.036279,1.931856,0.001287,7.555477,7.554190,1.230665,2.223021,4823.219233
1,001,CrossArms,RightWrist,CrossArms__RightWrist,001_preprocessed.npz,0.000139,-0.000414,0.076206,-0.287730,0.344297,0.632027,0.008399,0.076167,5.662223,0.000998,0.000138,0.111795,-0.404644,0.502851,0.907494,0.018903,0.111742,12.186582,-0.001049,-0.000991,0.094501,-0.331268,0.422170,0.753438,0.016056,0.094458,8.708211,0.108799,-0.000487,0.854009,-4.688707,3.174120,7.862827,0.048712,0.860477,722.651313,-0.146781,-0.004211,0.779226,-2.878691,2.977399,5.856090,0.087708,0.792537,613.040257,0.244460,0.007712,1.701535,-6.602785,5.189493,11.792278,0.056437,1.718143,2881.167400,0.096568,0.013777,0.133802,0.000801,0.573284,0.572483,0.168726,0.164955,26.557016,1.022689,0.038048,1.810530,0.002150,7.615600,7.613449,1.020145,2.078594,4216.858971
2,001,DrinkGlas,LeftWrist,DrinkGlas__LeftWrist,001_preprocessed.npz,0.000321,0.002782,0.063836,-0.212264,0.528248,0.740512,0.029106,0.063805,3.973319,0.000626,-0.001435,0.116101,-1.214733,0.949936,2.164669,0.047671,0.116043,13.142812,0.000461,0.001234,0.073780,-0.621156,0.441438,1.062594,0.034112,0.073744,5.307635,-0.046671,-0.016215,1.057995,-9.541655,3.595302,13.136956,0.453937,1.058483,1093.496105,0.001160,-0.002554,0.559553,-2.927128,1.708632,4.635760,0.261852,0.559268,305.273751,0.019834,-0.000634,0.558695,-1.383440,1.942469,3.325909,0.357143,0.558761,304.720697,0.089710,0.062060,0.122240,0.000652,1.385179,1.384527,0.116630,0.151576,22.423766,0.836871,0.522552,1.022789,0.002307,10.018781,10.016473,1.260728,1.321128,1703.490553
3,001,DrinkGlas,RightWrist,DrinkGlas__RightWrist,001_preprocessed.npz,0.000390,0.000479,0.114052,-0.471591,0.464202,0.935793,0.062724,0.113994,12.682797,0.000927,-0.003297,0.161418,-0.527266,0.801557,1.328823,0.119997,0.161338,25.405347,-0.000503,0.003436,0.131078,-0.763645,0.441676,1.205321,0.089243,0.131012,16.752168,0.035964,0.052533,2.480

In [9]:
feature_table_overview = pd.DataFrame(
    {
        "Metric": [
            "patient_id",
            "motor tasks",
            "wrist locations",
            "recordings",
            "Metadata columns",
            "Time-domain features",
            "Total columns",
            "Extraction errors",
        ],
        "Value": [
            time_feature_table["patient_id"].nunique(),
            time_feature_table["task"].nunique(),
            time_feature_table["wrist"].nunique(),
            len(time_feature_table),
            len(metadata_columns),
            len(feature_columns),
            time_feature_table.shape[1],
            len(extraction_errors),
        ],
    }
)

feature_table_overview

,Metric,Value
0,patient_id,469
1,motor tasks,11
2,wrist locations,2
3,recordings,10318
4,Metadata columns,5
5,Time-domain features,72
6,Total columns,77
7,Extraction errors,0


#### 6. Create Task-Aware Participant-Level Feature Table

The recording-level time-domain features were reorganized into a participant-level table while preserving the neurological assessment task and wrist associated with each feature.

Each resulting feature name identifies its task, wrist, sensor channel, and time-domain statistic. This structure maintains one row per participant while preserving the task-specific information required

In [10]:
# =============================================================================
# Create task-aware feature identifiers
# =============================================================================

task_aware_table = time_feature_table.copy()

# Shorter wrist labels
task_aware_table["wrist_short"] = (
    task_aware_table["wrist"]
    .replace({
        "LeftWrist": "Left",
        "RightWrist": "Right",
    })
)

# Combine task and wrist
task_aware_table["task_wrist"] = (
    task_aware_table["task"]
    + "_"
    + task_aware_table["wrist_short"]
)

print(
    "Task-wrist combinations:",
    task_aware_table["task_wrist"].nunique()
)

assert (
    task_aware_table["task_wrist"].nunique()
    == EXPECTED_RECORDINGS_PER_PARTICIPANT
), (
    "Unexpected number of task-wrist combinations."
)

Task-wrist combinations: 22


In [11]:
# =============================================================================
# Create participant-level task-aware feature table
# =============================================================================

task_aware_feature_table = (
    task_aware_table[
        [
            "patient_id",
            "task_wrist",
            *feature_columns,
        ]
    ]
    .set_index(
        [
            "patient_id",
            "task_wrist",
        ]
    )
    .unstack("task_wrist")
)

In [12]:
# =============================================================================
# Standardize feature names
# =============================================================================

def shorten_time_feature_name(feature_name):

    replacements = {
        "Accelerometer_X": "AccX",
        "Accelerometer_Y": "AccY",
        "Accelerometer_Z": "AccZ",
        "Gyroscope_X": "GyroX",
        "Gyroscope_Y": "GyroY",
        "Gyroscope_Z": "GyroZ",
        "Acc_Magnitude": "AccMag",
        "Gyro_Magnitude": "GyroMag",
    }

    for original, short in replacements.items():

        feature_name = feature_name.replace(
            original,
            short,
        )

    return feature_name

In [13]:
# =============================================================================
# Flatten task-aware column names
# =============================================================================

task_aware_feature_table.columns = [
    f"{task_wrist}_{shorten_time_feature_name(feature)}"
    for feature, task_wrist
    in task_aware_feature_table.columns
]

task_aware_feature_table = (
    task_aware_feature_table
    .reset_index()
)

In [14]:
# =============================================================================
# Inspect task-aware feature table
# =============================================================================

print(
    "Task-aware feature table shape:",
    task_aware_feature_table.shape
)

print(
    "Participants:",
    task_aware_feature_table[
        "patient_id"
    ].nunique()
)

print(
    "Task-aware time-domain features:",
    task_aware_feature_table.shape[1] - 1
)

task_aware_feature_table.iloc[
    :5,
    :12,
]

Task-aware feature table shape: (469, 1585)
Participants: 469
Task-aware time-domain features: 1584


,patient_id,CrossArms_Left_AccX_Mean,CrossArms_Right_AccX_Mean,DrinkGlas_Left_AccX_Mean,DrinkGlas_Right_AccX_Mean,Entrainment_Left_AccX_Mean,Entrainment_Right_AccX_Mean,HoldWeight_Left_AccX_Mean,HoldWeight_Right_AccX_Mean,LiftHold_Left_AccX_Mean,LiftHold_Right_AccX_Mean,PointFinger_Left_AccX_Mean
0,001,-0.000278,0.000139,0.000321,0.000390,0.000133,-0.000149,1.428852e-04,0.000059,-0.000554,-0.000735,-0.000156
1,002,-0.000024,-0.000415,0.003220,0.000288,-0.000354,-0.000638,1.848333e-04,-0.000127,-0.000429,0.000079,-0.003790
2,003,0.000742,-0.001091,-0.000721,0.000872,-0.000234,-0.000184,6.338518e-07,-0.000125,-0.001485,-0.000587,0.001295
3,004,0.000627,-0.001188,-0.000406,0.000204,-0.000192,-0.000285,7.054171e-05,0.000231,-0.000454,0.000560,-0.000325
4,005,-0.000261,-0.001369,-0.000220,-0.002176,0.000140,-0.000013,-1.312143e-04,0.000340,0.000419,-0.000799,0.000011


In [15]:
task_aware_feature_table

patient_id  CrossArms_Left_AccX_Mean  CrossArms_Right_AccX_Mean  \
0          001                 -0.000278               1.385881e-04   
1          002                 -0.000024              -4.153911e-04   
2          003                  0.000742              -1.091160e-03   
3          004                  0.000627              -1.188353e-03   
4          005                 -0.000261              -1.369428e-03   
..         ...                       ...                        ...   
464        465                  0.000627              -1.932761e-04   
465        466                  0.000005              -8.710853e-07   
466        467                 -0.000007               4.611833e-04   
467        468                 -0.000931               8.785627e-04   
468        469                 -0.001313              -1.901929e-04   

     DrinkGlas_Left_AccX_Mean  DrinkGlas_Right_AccX_Mean  \
0                    0.000321                   0.000390   
1                    0.003220                   0.000288   
2                   -0.000721                   0.000872   
3                   -0.000406                   0.000204   
4                   -0.000220                  -0.002176   
..                        ...                        ...   
464                 -0.000141                  -0.000149   
465                  0.000430                  -0.000459   
466                  0.000501                  -0.000575   
467                  0.000077                  -0.000941   
468                  0.000192                  -0.000081   

     Entrainment_Left_AccX_Mean  Entrainment_Right_AccX_Mean  \
0                      0.000133                    -0.000149   
1                     -0.000354                    -0.000638   
2                     -0.000234                    -0.000184   
3                     -0.000192                    -0.000285   
4                      0.000140                    -0.000013   
..                          ...                          ...   
464                   -0.000897                    -0.000740   
465                   -0.000024                     0.000161   
466                    0.000036                     0.000140   
467                   -0.000148                    -0.000082   
468                    0.000195                    -0.000115   

     HoldWeight_Left_AccX_Mean  HoldWeight_Right_AccX_Mean  \
0                 1.428852e-04                    0.000059   
1                 1.848333e-04                   -0.000127   
2                 6.338518e-07                   -0.000125   
3                 7.054171e-05                    0.000231   
4                -1.312143e-04                    0.000340   
..                         ...                         ...   
464              -3.174780e-04                   -0.000434   
465               3.440709e-04                    0.000219   
466               1.104497e-04                   -0.000278   
467              -2.456843e-04                   -0.000266   
468              -3.981110e-04                   -0.000278   

     LiftHold_Left_AccX_Mean  LiftHold_Right_AccX_Mean  \
0                  -0.000554                 -0.000735   
1                  -0.000429                  0.000079   
2                  -0.001485                 -0.000587   
3                  -0.000454                  0.000560   
4                   0.000419                 -0.000799   
..                       ...                       ...   
464                 0.000271                  0.000016   
465                 0.000081                  0.000147   
466                 0.000722                 -0.000559   
467                -0.000210                 -0.000156   
468                 0.000937                 -0.000759   

     PointFinger_Left_AccX_Mean  PointFinger_Right_AccX_Mean  \
0                     -0.000156                     0.000249   
1                     -0.003790                     0.002324   
2                      0.00

In [16]:
# =============================================================================
# Validate task-aware feature table
# =============================================================================

EXPECTED_TASK_AWARE_TIME_FEATURES = (
    EXPECTED_TIME_FEATURES
    * EXPECTED_RECORDINGS_PER_PARTICIPANT
)

assert len(
    task_aware_feature_table
) == EXPECTED_PARTICIPANTS, (
    f"Expected {EXPECTED_PARTICIPANTS} participants, "
    f"but found {len(task_aware_feature_table)}."
)

assert (
    task_aware_feature_table[
        "patient_id"
    ].nunique()
    == EXPECTED_PARTICIPANTS
), (
    "Unexpected duplicated or missing participant IDs."
)

assert (
    task_aware_feature_table.shape[1] - 1
    == EXPECTED_TASK_AWARE_TIME_FEATURES
), (
    f"Expected {EXPECTED_TASK_AWARE_TIME_FEATURES} "
    f"task-aware features, but found "
    f"{task_aware_feature_table.shape[1] - 1}."
)

print(
    "Task-aware Time-Domain Feature table verfied"
)

Task-aware Time-Domain Feature table verfied


#### 7. Feature quality assessment

Feature quality was evaluated at two levels.

First, the 72 original time-domain feature types were assessed across all 10,318 task–wrist recordings for missing values, infinite values, uniqueness, variance, constant behavior, low variance, and invalid numerical values.

Second, the participant-level task-aware time-domain feature table was evaluated to confirm that the restructuring process preserved complete and valid numerical feature values.


#### 7.1 Recording-Level Feature Quality

In [17]:
# ==================================================
# Select numerical time-domain feature columns
# ==================================================

numeric_features = (
    time_feature_table[feature_columns]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

print("Numeric feature matrix shape:", numeric_features.shape)

Numeric feature matrix shape: (10318, 72)


In [18]:
# =============================================
# Calculate feature-level quality metrics
# =============================================

missing_counts = (
    numeric_features
    .isna()
    .sum()
)

infinite_counts = pd.Series(
    np.isinf(
        numeric_features.to_numpy(dtype=float)
    ).sum(axis=0),
    index=numeric_features.columns,
)

feature_variances = numeric_features.var(ddof=1)

unique_counts = numeric_features.nunique(dropna=True)

constant_flags = (unique_counts <= 1)

low_variance_flags = (
    (~constant_flags)
    & feature_variances.notna()
    & (feature_variances < LOW_VARIANCE_THRESHOLD)
)

invalid_flags = (
    (missing_counts > 0)
    | (infinite_counts > 0)
)

recording_feature_quality_report = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Missing Values": missing_counts.reindex(
            feature_columns
        ).to_numpy(),
        "Infinite Values": infinite_counts.reindex(
            feature_columns
        ).to_numpy(),
        "Unique Values": unique_counts.reindex(
            feature_columns
        ).to_numpy(),
        "Variance": feature_variances.reindex(
            feature_columns
        ).to_numpy(),
        "Constant": constant_flags.reindex(
            feature_columns
        ).to_numpy(),
        "Low Variance": low_variance_flags.reindex(
            feature_columns
        ).to_numpy(),
        "Invalid": invalid_flags.reindex(
            feature_columns
        ).to_numpy(),
    }
)

recording_feature_quality_report.head(15)

,Feature,Missing Values,Infinite Values,Unique Values,Variance,Constant,Low Variance,Invalid
0,Accelerometer_X_Mean,0,0,10318,6.632959e-07,False,True,False
1,Accelerometer_X_Median,0,0,10318,7.453210e-05,False,False,False
2,Accelerometer_X_Std,0,0,10318,4.083710e-03,False,False,False
3,Accelerometer_X_Min,0,0,10318,1.403440e-01,False,False,False
4,Accelerometer_X_Max,0,0,10318,1.252012e-01,False,False,False
5,Accelerometer_X_Range,0,0,10318,4.461665e-01,False,False,False
6,Accelerometer_X_IQR,0,0,10318,5.677024e-03,False,False,False
7,Accelerometer_X_RMS,0,0,10318,4.079591e-03,False,False,False
8,Accelerometer_X_Energy,0,0,10318,4.778873e+02,False,False,False
9,Accelerometer_Y_Mean,0,0,10318,8.641696e-07,False,True,False


#### 7.2 Task-Aware Feature Quality

In [19]:
# =============================================================================
# Select task-aware predictor columns
# =============================================================================

task_aware_feature_columns = [
    column
    for column in task_aware_feature_table.columns
    if column != "patient_id"
]

task_aware_numeric = (
    task_aware_feature_table[
        task_aware_feature_columns
    ]
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
)

print(
    "Task-aware numeric feature matrix shape:",
    task_aware_numeric.shape
)

Task-aware numeric feature matrix shape: (469, 1584)


In [20]:
# =============================================================================
# Calculate task-aware feature quality metrics
# =============================================================================

task_aware_missing_counts = (
    task_aware_numeric
    .isna()
    .sum()
)

task_aware_infinite_counts = pd.Series(
    np.isinf(
        task_aware_numeric
        .to_numpy(dtype=float)
    ).sum(axis=0),
    index=task_aware_numeric.columns,
)

task_aware_variances = (
    task_aware_numeric
    .var(ddof=1)
)

task_aware_unique_counts = (
    task_aware_numeric
    .nunique(
        dropna=True
    )
)

task_aware_constant_flags = (
    task_aware_unique_counts <= 1
)

task_aware_low_variance_flags = (
    (~task_aware_constant_flags)
    & task_aware_variances.notna()
    & (
        task_aware_variances
        < LOW_VARIANCE_THRESHOLD
    )
)

task_aware_invalid_flags = (
    (task_aware_missing_counts > 0)
    | (task_aware_infinite_counts > 0)
)

In [21]:
# =============================================================================
# Create task-aware feature quality report
# =============================================================================

task_aware_feature_quality_report = pd.DataFrame(
    {
        "Feature": task_aware_feature_columns,

        "Missing Values":
            task_aware_missing_counts
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Infinite Values":
            task_aware_infinite_counts
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Unique Values":
            task_aware_unique_counts
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Variance":
            task_aware_variances
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Constant":
            task_aware_constant_flags
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Low Variance":
            task_aware_low_variance_flags
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Invalid":
            task_aware_invalid_flags
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),
    }
)

task_aware_feature_quality_report.head(15)

,Feature,Missing Values,Infinite Values,Unique Values,Variance,Constant,Low Variance,Invalid
0,CrossArms_Left_AccX_Mean,0,0,469,9.377872e-07,False,True,False
1,CrossArms_Right_AccX_Mean,0,0,469,8.655441e-07,False,True,False
2,DrinkGlas_Left_AccX_Mean,0,0,469,7.705555e-07,False,True,False
3,DrinkGlas_Right_AccX_Mean,0,0,469,9.804176e-07,False,True,False
4,Entrainment_Left_AccX_Mean,0,0,469,8.779158e-08,False,True,False
5,Entrainment_Right_AccX_Mean,0,0,469,9.029876e-08,False,True,False
6,HoldWeight_Left_AccX_Mean,0,0,469,1.416459e-07,False,True,False
7,HoldWeight_Right_AccX_Mean,0,0,469,1.354196e-07,False,True,False
8,LiftHold_Left_AccX_Mean,0,0,469,3.582144e-07,False,True,False
9,LiftHold_Right_AccX_Mean,0,0,469,2.934193e-07,False,True,False


In [22]:
# =============================================================================
# Create task-aware feature quality report
# =============================================================================

task_aware_feature_quality_report = pd.DataFrame(
    {
        "Feature":
            task_aware_feature_columns,

        "Missing Values":
            task_aware_missing_counts
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Infinite Values":
            task_aware_infinite_counts
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Unique Values":
            task_aware_unique_counts
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Variance":
            task_aware_variances
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Constant":
            task_aware_constant_flags
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Low Variance":
            task_aware_low_variance_flags
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),

        "Invalid":
            task_aware_invalid_flags
            .reindex(
                task_aware_feature_columns
            )
            .to_numpy(),
    }
)

task_aware_feature_quality_report.head(15)

,Feature,Missing Values,Infinite Values,Unique Values,Variance,Constant,Low Variance,Invalid
0,CrossArms_Left_AccX_Mean,0,0,469,9.377872e-07,False,True,False
1,CrossArms_Right_AccX_Mean,0,0,469,8.655441e-07,False,True,False
2,DrinkGlas_Left_AccX_Mean,0,0,469,7.705555e-07,False,True,False
3,DrinkGlas_Right_AccX_Mean,0,0,469,9.804176e-07,False,True,False
4,Entrainment_Left_AccX_Mean,0,0,469,8.779158e-08,False,True,False
5,Entrainment_Right_AccX_Mean,0,0,469,9.029876e-08,False,True,False
6,HoldWeight_Left_AccX_Mean,0,0,469,1.416459e-07,False,True,False
7,HoldWeight_Right_AccX_Mean,0,0,469,1.354196e-07,False,True,False
8,LiftHold_Left_AccX_Mean,0,0,469,3.582144e-07,False,True,False
9,LiftHold_Right_AccX_Mean,0,0,469,2.934193e-07,False,True,False


#### 8. List constant, invalid and low-variance features

In [23]:
# =============================================================================
# Constant task-aware features
# =============================================================================

task_aware_constant_features = (
    task_aware_feature_quality_report[
        task_aware_feature_quality_report[
            "Constant"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Task-aware constant features:",
    len(task_aware_constant_features)
)

task_aware_constant_features

Task-aware constant features: 0


,Feature,Missing Values,Infinite Values,Unique Values,Variance,Constant,Low Variance,Invalid


In [24]:
# =============================================================================
# Invalid task-aware features
# =============================================================================

task_aware_invalid_features = (
    task_aware_feature_quality_report[
        task_aware_feature_quality_report[
            "Invalid"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Task-aware invalid features:",
    len(task_aware_invalid_features)
)

task_aware_invalid_features

Task-aware invalid features: 0


,Feature,Missing Values,Infinite Values,Unique Values,Variance,Constant,Low Variance,Invalid


In [25]:
# =============================================================================
# Low-variance task-aware features
# =============================================================================

task_aware_low_variance_features = (
    task_aware_feature_quality_report[
        task_aware_feature_quality_report[
            "Low Variance"
        ]
    ]
    .copy()
    .sort_values(
        "Variance",
        ascending=True,
    )
    .reset_index(drop=True)
)

print(
    "Task-aware low-variance features:",
    len(task_aware_low_variance_features)
)

task_aware_low_variance_features

Task-aware low-variance features: 45


,Feature,Missing Values,Infinite Values,Unique Values,Variance,Constant,Low Variance,Invalid
0,Relaxed_Right_AccZ_Mean,0,0,469,6.116775e-09,False,True,False
1,RelaxedTask_Right_AccZ_Mean,0,0,469,9.001532e-09,False,True,False
2,RelaxedTask_Left_AccZ_Mean,0,0,469,9.577027e-09,False,True,False
3,Relaxed_Right_AccY_Mean,0,0,469,1.101758e-08,False,True,False
4,Relaxed_Left_AccZ_Mean,0,0,469,1.154505e-08,False,True,False
5,RelaxedTask_Right_AccY_Mean,0,0,469,1.267314e-08,False,True,False
6,Relaxed_Left_AccX_Mean,0,0,469,1.280401e-08,False,True,False
7,Relaxed_Right_AccX_Mean,0,0,469,1.481217e-08,False,True,False
8,RelaxedTask_Left_AccX_Mean,0,0,469,1.707431e-08,False,True,False
9,RelaxedTask_Right_AccX_Mean,0,0,469,1.885157e-08,False,True,False


#### 9. Task-Aware Feature-quality summary

In [26]:
# =============================================================================
# Create task-aware feature-quality summary
# =============================================================================

total_task_aware_missing = int(
    task_aware_numeric
    .isna()
    .sum()
    .sum()
)

total_task_aware_infinite = int(
    np.isinf(
        task_aware_numeric
        .to_numpy(dtype=float)
    ).sum()
)

task_aware_features_with_missing = int(
    (task_aware_missing_counts > 0).sum()
)

task_aware_features_with_infinite = int(
    (task_aware_infinite_counts > 0).sum()
)

task_aware_feature_quality_summary = pd.DataFrame(
    {
        "Metric": [
            "Participants",
            "Task-aware time-domain features",
            "Expected task-aware features",
            "Total missing values",
            "Features with missing values",
            "Total infinite values",
            "Features with infinite values",
            "Constant features",
            "Low-variance features",
            "Invalid features",
        ],
        "Value": [
            len(task_aware_feature_table),
            len(task_aware_feature_columns),
            EXPECTED_TASK_AWARE_TIME_FEATURES,
            total_task_aware_missing,
            task_aware_features_with_missing,
            total_task_aware_infinite,
            task_aware_features_with_infinite,
            len(task_aware_constant_features),
            len(task_aware_low_variance_features),
            len(task_aware_invalid_features),
        ],
    }
)

task_aware_feature_quality_summary

,Metric,Value
0,Participants,469
1,Task-aware time-domain features,1584
2,Expected task-aware features,1584
3,Total missing values,0
4,Features with missing values,0
5,Total infinite values,0
6,Features with infinite values,0
7,Constant features,0
8,Low-variance features,45
9,Invalid features,0


In [27]:
# =============================================================================
# Validate task-aware feature quality
# =============================================================================

assert total_task_aware_missing == 0, (
    "Task-aware feature table contains missing values."
)

assert total_task_aware_infinite == 0, (
    "Task-aware feature table contains infinite values."
)

assert len(task_aware_invalid_features) == 0, (
    "Invalid task-aware features were detected."
)

print(
    "Task-Aware Feature Quality validation passed"
)

Task-Aware Feature Quality validation passed


#### 10. Feature completeness result

In [28]:
# ========================================
# Recording-level feature completeness
# ========================================

feature_completeness_df = time_feature_table[
    metadata_columns
].copy()

feature_completeness_df[
    "Available Features"
] = (
    numeric_features
    .notna()
    .sum(axis=1)
)

feature_completeness_df[
    "Expected Features"
] = EXPECTED_TIME_FEATURES

feature_completeness_df[
    "Complete"
] = (
    feature_completeness_df["Available Features"]
    == feature_completeness_df["Expected Features"]
)

feature_completeness_df.head()

,patient_id,task,wrist,recording_key,source_file,Available Features,Expected Features,Complete
0,001,CrossArms,LeftWrist,CrossArms__LeftWrist,001_preprocessed.npz,72,72,True
1,001,CrossArms,RightWrist,CrossArms__RightWrist,001_preprocessed.npz,72,72,True
2,001,DrinkGlas,LeftWrist,DrinkGlas__LeftWrist,001_preprocessed.npz,72,72,True
3,001,DrinkGlas,RightWrist,DrinkGlas__RightWrist,001_preprocessed.npz,72,72,True
4,001,Entrainment,LeftWrist,Entrainment__LeftWrist,001_preprocessed.npz,72,72,True


In [29]:
feature_completeness_summary = pd.DataFrame(
    {
        "Metric": [
            "Total recordings",
            "Complete feature vectors",
            "Incomplete feature vectors",
            "Overall feature completeness (%)",
        ],
        "Value": [
            len(feature_completeness_df),
            int(
                feature_completeness_df[
                    "Complete"
                ].sum()
            ),
            int(
                (
                    ~feature_completeness_df[
                        "Complete"
                    ]
                ).sum()
            ),
            round(
                feature_completeness_df[
                    "Complete"
                ].mean()
                * 100,
                2,
            ),
        ],
    }
)

feature_completeness_summary

,Metric,Value
0,Total recordings,10318.0
1,Complete feature vectors,10318.0
2,Incomplete feature vectors,0.0
3,Overall feature completeness (%),100.0


#### 11. Save and verify outputs

In [30]:
# ================================================
# Save recording level Time-Domain table outputs
# ================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

time_feature_table.to_csv(
    RECORDING_TIME_FEATURES_OUTPUT,
    index=False,
)

recording_feature_quality_report.to_csv(
    RECORDING_FEATURE_QUALITY_REPORT_OUTPUT,
    index=False,
)

feature_completeness_df.to_csv(
    FEATURE_COMPLETENESS_OUTPUT,
    index=False,
)

feature_completeness_summary.to_csv(
    FEATURE_COMPLETENESS_SUMMARY_OUTPUT,
    index=False,
)

# =============================================================================
# Save task-aware outputs
# =============================================================================

task_aware_feature_table.to_csv(
    TIME_FEATURES_OUTPUT,
    index=False,
)

task_aware_feature_quality_report.to_csv(
    TASK_AWARE_FEATURE_QUALITY_REPORT_OUTPUT,
    index=False,
)

task_aware_feature_quality_summary.to_csv(
    TASK_AWARE_FEATURE_QUALITY_SUMMARY_OUTPUT,
    index=False,
)

task_aware_constant_features.to_csv(
    TASK_AWARE_CONSTANT_FEATURES_OUTPUT,
    index=False,
)

task_aware_invalid_features.to_csv(
    TASK_AWARE_INVALID_FEATURES_OUTPUT,
    index=False,
)

task_aware_low_variance_features.to_csv(
    TASK_AWARE_LOW_VARIANCE_FEATURES_OUTPUT,
    index=False,
)


# =============================================================================
# Verify that files exist and are not empty
# =============================================================================

output_files = {
    "Recording-level Time-domain feature table":
        RECORDING_TIME_FEATURES_OUTPUT,

    "Feature-quality report":
        RECORDING_FEATURE_QUALITY_REPORT_OUTPUT,

    "Task-aware time-domain feature table":
        TIME_FEATURES_OUTPUT,

    "Recording-level feature-quality report":
        RECORDING_FEATURE_QUALITY_REPORT_OUTPUT,

    "Task-aware feature-quality report":
        TASK_AWARE_FEATURE_QUALITY_REPORT_OUTPUT,

    "Task-aware feature-quality summary":
        TASK_AWARE_FEATURE_QUALITY_SUMMARY_OUTPUT,

    "Task-aware constant features":
        TASK_AWARE_CONSTANT_FEATURES_OUTPUT,

    "Task-aware invalid features":
        TASK_AWARE_INVALID_FEATURES_OUTPUT,

    "Task-aware low-variance features":
        TASK_AWARE_LOW_VARIANCE_FEATURES_OUTPUT,
}

for output_name, output_path in output_files.items():

    assert output_path.exists(), (
        f"{output_name} was not saved: {output_path}"
    )

    assert output_path.stat().st_size > 0, (
        f"{output_name} is empty: {output_path}"
    )

print("All output files exist and are non-empty.")

All output files exist and are non-empty.


In [31]:
# ================================
# Read the files again
# ================================

recording_time_features_check = pd.read_csv(
    RECORDING_TIME_FEATURES_OUTPUT,
    dtype={
        "patient_id": str,
    },
)

quality_report_check = pd.read_csv(
    RECORDING_FEATURE_QUALITY_REPORT_OUTPUT
)

time_features_check = pd.read_csv(
    TIME_FEATURES_OUTPUT,
    dtype={
        "patient_id": str,
    },
)

recording_quality_report_check = pd.read_csv(
    RECORDING_FEATURE_QUALITY_REPORT_OUTPUT
)

task_aware_quality_report_check = pd.read_csv(
    TASK_AWARE_FEATURE_QUALITY_REPORT_OUTPUT
)

task_aware_quality_summary_check = pd.read_csv(
    TASK_AWARE_FEATURE_QUALITY_SUMMARY_OUTPUT
)

task_aware_constant_features_check = pd.read_csv(
    TASK_AWARE_CONSTANT_FEATURES_OUTPUT
)

task_aware_invalid_features_check = pd.read_csv(
    TASK_AWARE_INVALID_FEATURES_OUTPUT
)

task_aware_low_variance_features_check = pd.read_csv(
    TASK_AWARE_LOW_VARIANCE_FEATURES_OUTPUT
)

# ===================================
# Verificar recording-level output
# ===================================
assert (
    recording_time_features_check.shape
    == time_feature_table.shape
), (
    "Recording-level time-domain feature table "
    "shape changed during export."
)

assert (
    list(recording_time_features_check.columns)
    == list(time_feature_table.columns)
), (
    "Recording-level time-domain feature columns "
    "changed during export."
)

# =============================================================================
# Verify task-aware output
# =============================================================================

assert (
    time_features_check.shape
    == task_aware_feature_table.shape
), (
    "Task-aware time-domain feature table "
    "shape changed during export."
)

assert (
    list(time_features_check.columns)
    == list(task_aware_feature_table.columns)
), (
    "Task-aware feature columns changed during export."
)

# =============================================================================
# Verify feature-quality reports
# =============================================================================

assert (
    recording_quality_report_check.shape
    == recording_feature_quality_report.shape
), (
    "Recording-level feature-quality report "
    "shape changed during export."
)

assert (
    list(recording_quality_report_check.columns)
    == list(recording_feature_quality_report.columns)
), (
    "Recording-level feature-quality report "
    "columns changed during export."
)


assert (
    task_aware_quality_report_check.shape
    == task_aware_feature_quality_report.shape
), (
    "Task-aware feature-quality report "
    "shape changed during export."
)

assert (
    list(task_aware_quality_report_check.columns)
    == list(task_aware_feature_quality_report.columns)
), (
    "Task-aware feature-quality report "
    "columns changed during export."
)

# =============================================================================
# Verify task-aware quality summary
# =============================================================================

assert (
    task_aware_quality_summary_check.shape
    == task_aware_feature_quality_summary.shape
), (
    "Task-aware feature-quality summary "
    "shape changed during export."
)

# =============================================================================
# Verify task-aware feature lists
# =============================================================================

assert (
    task_aware_constant_features_check.shape
    == task_aware_constant_features.shape
), (
    "Task-aware constant-feature table "
    "shape changed during export."
)

assert (
    task_aware_invalid_features_check.shape
    == task_aware_invalid_features.shape
), (
    "Task-aware invalid-feature table "
    "shape changed during export."
)

assert (
    task_aware_low_variance_features_check.shape
    == task_aware_low_variance_features.shape
), (
    "Task-aware low-variance feature table "
    "shape changed during export."
)

In [32]:
# =============================================================================
# Final save verification summary
# =============================================================================

print("SAVE VERIFICATION PASSED\n")

for output_name, output_path in output_files.items():
    print(
        f"{output_name}: {output_path}"
    )

print()

print(
    "Recording-level records verified:",
    f"{len(recording_time_features_check):,}",
)

print(
    "Recording-level feature types verified:",
    len(feature_columns),
)

print(
    "Task-aware participants verified:",
    len(time_features_check),
)

print(
    "Task-aware feature columns verified:",
    len(task_aware_feature_columns),
)

print(
    "Recording-level quality-report rows verified:",
    len(recording_quality_report_check),
)

print(
    "Task-aware quality-report rows verified:",
    len(task_aware_quality_report_check),
)

SAVE VERIFICATION PASSED

Recording-level Time-domain feature table: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\time_domain_features_recording_level.csv
Feature-quality report: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\tables\time_domain_features\recording_feature_quality_report.csv
Task-aware time-domain feature table: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\time_domain_features.csv
Recording-level feature-quality report: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\tables\time_domain_features\recording_feature_quality_report.csv
Task-aware feature-quality report: c:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\outputs\tables\time_domain_features\task_aware_feature_quality_report.csv
Tas

#### Conclusion:

The time-domain feature extraction and quality assessment were successfully completed for all 469 participants. A total of 10,318 task–wrist recordings were processed, with 72 time-domain features extracted from each recording and no extraction errors.

The recording-level features were reorganized into a task-aware participant-level dataset containing 1,584 predictors per participant. This structure preserves the neurological assessment task and wrist associated with each movement feature while maintaining one row per participant
